In [2]:
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score,precision_recall_curve

In [3]:
df=pd.read_csv(r'dataset\dataset_ampicillin_final.csv')

In [4]:
df

,Ampicillin_Resistance,Gene_blaTEM-1,Gene_blaTEM-135,Gene_blaOXA-1,Gene_blaCMY-2,Gene_blaCTX-M-15,Gene_blaEC,Gene_mdtM,Gene_acrF,Gene_emrD,Gene_ompF,Gene_ompC
0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
20404,1,0,0,0,0,0,0,0,0,0,0,0
20405,0,0,0,0,0,0,0,0,0,0,0,0
20406,1,0,0,0,0,0,0,0,0,0,0,0
20407,1,0,0,0,0,0,0,0,0,0,0,0


In [5]:
X=df.iloc[:,1:]

In [6]:
y=df.iloc[:,0]

In [7]:
X_train,X_test,y_train,y_test=tts(X,y,test_size=0.2,random_state=42)

In [8]:
from sklearn.model_selection import RandomizedSearchCV
rf=ExtraTreesClassifier(random_state=42)
rf_params = {
    "n_estimators": [50, 100, 200, 300, 400, 500],
    "max_depth": [None, 5,10,20, 40,60],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4,10],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True]
}
search = RandomizedSearchCV(
    rf,
    rf_params,
    n_iter=20,
    refit="accuracy",
    n_jobs=-1,
    cv=5,
    scoring=['accuracy', 'f1', 'roc_auc','recall'],
)

a=search.fit(X_train, y_train)
a

RandomizedSearchCV(cv=5, estimator=ExtraTreesClassifier(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'bootstrap': [True],
                                        'max_depth': [None, 5, 10, 20, 40, 60],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4, 10],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200, 300, 400,
                                                         500]},
                   refit='accuracy',
                   scoring=['accuracy', 'f1', 'roc_auc', 'recall'])

In [9]:
et=ExtraTreesClassifier(bootstrap=True, max_depth=5, random_state=42)

In [10]:
et.fit(X_train,y_train)
y_pred=et.predict(X_test)

In [11]:
accuracy_score(y_pred,y_test)

0.8701616854483096

In [12]:
f1_score(y_pred,y_test)

0.8205822613405552

In [13]:
roc_auc_score(y_pred,y_test)

np.float64(0.8844728047190293)

In [15]:
import pandas as pd

# 1. Define your exact feature names (Order matters!)
# This matches the columns in your uploaded CSV.
feature_columns = [
    'Gene_blaTEM-1', 
    'Gene_blaTEM-135', 
    'Gene_blaOXA-1', 
    'Gene_blaCMY-2', 
    'Gene_blaCTX-M-15', 
    'Gene_blaEC', 
    'Gene_mdtM', 
    'Gene_acrF', 
    'Gene_emrD', 
    'Gene_ompF', 
    'Gene_ompC'
]

# 2. Create the Test Scenarios (Rows of data)
test_data = [
    # Case 1: The "Classic" Resistant (Has blaTEM-1)
    # This is the #1 cause of Ampicillin resistance. Should be RESISTANT.
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

    # Case 2: The "Clean" Sample (No genes)
    # Should be SENSITIVE.
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

    # Case 3: The "Superbug" (Has blaCTX-M-15)
    # This is an ESBL gene. Should be RESISTANT.
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],

    # Case 4: The "AmpC" Resistant (Has blaCMY-2)
    # A different mechanism. Should be RESISTANT.
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],

    # Case 5: Pump Only (Has Gene_mdtM)
    # This tests if a pump alone triggers resistance. Result may vary (Likely Sensitive or Low Prob).
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
]

# 3. Convert to DataFrame
test_df = pd.DataFrame(test_data, columns=feature_columns)

# 4. Run Prediction
print("\n--- 🧪 LIVE MODEL TEST RESULTS ---")
predictions = et.predict(test_df)
probabilities =et.predict_proba(test_df)[:, 1]

scenarios = [
    "Classic Resistant (blaTEM-1)", 
    "Clean/Sensitive (No Genes)", 
    "Superbug (blaCTX-M-15)", 
    "AmpC Resistant (blaCMY-2)", 
    "Pump Only (mdtM)"
]

for i, scenario in enumerate(scenarios):
    result = "🔴 RESISTANT" if predictions[i] == 1 else "🟢 SENSITIVE"
    print(f"Test {i+1}: {scenario:<30} -> {result} (Confidence: {probabilities[i]:.2%})")


--- 🧪 LIVE MODEL TEST RESULTS ---
Test 1: Classic Resistant (blaTEM-1)   -> 🔴 RESISTANT (Confidence: 83.09%)
Test 2: Clean/Sensitive (No Genes)     -> 🟢 SENSITIVE (Confidence: 18.29%)
Test 3: Superbug (blaCTX-M-15)         -> 🔴 RESISTANT (Confidence: 69.20%)
Test 4: AmpC Resistant (blaCMY-2)      -> 🔴 RESISTANT (Confidence: 81.51%)
Test 5: Pump Only (mdtM)               -> 🟢 SENSITIVE (Confidence: 20.83%)


In [16]:
import pandas as pd
import random
from sklearn.metrics import accuracy_score

# 1. Define Features (Must match your model's 11 inputs)
feature_columns = [
    'Gene_blaTEM-1', 'Gene_blaTEM-135', 'Gene_blaOXA-1', 'Gene_blaCMY-2', 
    'Gene_blaCTX-M-15', 'Gene_blaEC', 'Gene_mdtM', 'Gene_acrF', 
    'Gene_emrD', 'Gene_ompF', 'Gene_ompC'
]

# 2. Generate 50 Synthetic Test Samples
# We create patterns based on biological rules to see if the AI learned them.
test_data = []
expected_labels = [] # What we EXPECT the model to say (Ground Truth)

for i in range(50):
    row = [0] * 11 # Start with all 0s
    
    # --- Scenario A: The "Killer" Genes (Should be Resistant) ---
    if i < 20: 
        # Inject a strong resistance gene
        killer_gene_idx = random.choice([0, 3, 4]) # blaTEM-1, blaCMY-2, blaCTX-M-15
        row[killer_gene_idx] = 1
        
        # Add random noise (maybe a pump exists too)
        if random.random() > 0.5: row[6] = 1 # mdtM
        
        test_data.append(row)
        expected_labels.append(1) # Expect RESISTANT
        
    # --- Scenario B: The "Clean" Samples (Should be Sensitive) ---
    elif i < 35:
        # No genes at all, or just harmless random noise
        test_data.append(row)
        expected_labels.append(0) # Expect SENSITIVE
        
    # --- Scenario C: The "Weak" Resistance (Pumps Only) ---
    elif i < 45:
        # Only pumps (mdtM, acrF, emrD) - usually not enough for full resistance
        pump_idx = random.choice([6, 7, 8]) 
        row[pump_idx] = 1
        test_data.append(row)
        expected_labels.append(0) # Expect SENSITIVE (usually)
        
    # --- Scenario D: The "Complex" Mix (Multiple Genes) ---
    else:
        # blaTEM-1 + blaEC + Pump
        row[0] = 1 # blaTEM-1
        row[5] = 1 # blaEC
        row[8] = 1 # emrD
        test_data.append(row)
        expected_labels.append(1) # Expect RESISTANT

# 3. Convert to DataFrame
test_df = pd.DataFrame(test_data, columns=feature_columns)

# 4. Run Model Prediction
predictions = et.predict(test_df)
probs = et.predict_proba(test_df)[:, 1]

# 5. Generate Report
print(f"--- 📊 AUTOMATED 50-SAMPLE TEST REPORT ---")
print(f"Test Set Size: 50 Samples")
print(f"-"*60)
print(f"{'ID':<4} | {'Scenario Type':<20} | {'Genes Present':<30} | {'Predicted':<10} | {'Conf.%':<6}")
print(f"-"*60)

correct_count = 0
for i in range(50):
    # Determine type for display
    if i < 20: type_lbl = "Strong Resistance"
    elif i < 35: type_lbl = "Clean / Sensitive"
    elif i < 45: type_lbl = "Weak (Pump Only)"
    else: type_lbl = "Complex Mix"
    
    # Get active genes for display
    active_genes = [col.replace('Gene_', '') for col, val in zip(feature_columns, test_data[i]) if val == 1]
    genes_str = ", ".join(active_genes) if active_genes else "None"
    
    # Check correctness (loosely based on our expectation)
    pred_label = "RESISTANT" if predictions[i] == 1 else "SENSITIVE"
    is_match = (predictions[i] == expected_labels[i])
    if is_match: correct_count += 1
    
    # Print only first 10 and last 5 to save space, or ALL if you prefer
    # Let's print a selection to show variety
    if i < 5 or (i >= 20 and i < 25) or (i >= 35 and i < 40) or i >= 45:
        print(f"{i+1:<4} | {type_lbl:<20} | {genes_str[:28]:<30} | {pred_label:<10} | {probs[i]:.2f}")

print(f"...\n(Showing sample rows. Total processed: 50)")
print(f"-"*60)

# 6. Final Score
score = (correct_count / 50) * 100
print(f"✅ PASSED: {correct_count}/50 Test Cases")
print(f"🏆 SYNTHETIC ACCURACY SCORE: {score:.1f}%")

if score > 85:
    print("\n🌟 CONCLUSION: Model behavior is scientifically sound!")
else:
    print("\n⚠️ CONCLUSION: Model behavior needs review (check Pump predictions).")

--- 📊 AUTOMATED 50-SAMPLE TEST REPORT ---
Test Set Size: 50 Samples
------------------------------------------------------------
ID   | Scenario Type        | Genes Present                  | Predicted  | Conf.%
------------------------------------------------------------
1    | Strong Resistance    | blaCMY-2                       | RESISTANT  | 0.82
2    | Strong Resistance    | blaCMY-2, mdtM                 | RESISTANT  | 0.81
3    | Strong Resistance    | blaCMY-2, mdtM                 | RESISTANT  | 0.81
4    | Strong Resistance    | blaCMY-2, mdtM                 | RESISTANT  | 0.81
5    | Strong Resistance    | blaCMY-2, mdtM                 | RESISTANT  | 0.81
21   | Clean / Sensitive    | None                           | SENSITIVE  | 0.18
22   | Clean / Sensitive    | None                           | SENSITIVE  | 0.18
23   | Clean / Sensitive    | None                           | SENSITIVE  | 0.18
24   | Clean / Sensitive    | None                           | SENSITIVE  | 0.1